# 02 · Paraxial frente a vectorial y aberración esférica

Actividades 4 y 5 (entregables E5 y E6) y los extras de forma de lente y caras asféricas. Aquí se calculan todos los números que cita [`docs/aberraciones.md`](../docs/aberraciones.md); las figuras son fig02, fig03 y fig04.

Lente de trabajo: biconvexa simétrica con radios de vértice $R = 10$, espesor $2$ y $n_L = 1.5$; objeto sobre el eje en $y = -30$. Se compara con caras parabólicas ($k = -1$) y esféricas ($k = 0$). La pupila completa llega hasta $h = 3.92$, el 90 % del borde de la lente esférica, y es la misma en los notebooks 02 y 04.

In [1]:
import numpy as np
from scipy.optimize import minimize_scalar

from lentes import (focal_efectiva, imagen_lente_delgada, imagen_paraxial, lente_biconvexa, lente_con_forma,
                    planos_principales)
from lentes.aberraciones import (aberracion_longitudinal, coeficiente_esferica_trazado,
                                 mejores_focos, y_paraxial)
from lentes.difraccion import apertura_limite, haz
from lentes.seidel import coeficiente_esferica, k_que_anula

Y_OBJ = -30.0
parab = lente_biconvexa(10, 10, 2.0, 1.5, -1.0, -1.0)
esfer = lente_biconvexa(10, 10, 2.0, 1.5, 0.0, 0.0)
H_MAX = 0.9 * min(parab.semiapertura, esfer.semiapertura)
print(f"Imagen paraxial (las dos lentes): y = {y_paraxial(parab, Y_OBJ):.4f}")
print(f"Pupila completa: rayos hasta h = {H_MAX:.2f}")

Imagen paraxial (las dos lentes): y = 16.1872
Pupila completa: rayos hasta h = 3.92


## 1. Teoría de tercer orden contra el trazado exacto

Para $h$ pequeño, $\Delta y \approx C_3 h^2$. El trazado mide $C_3$ ajustando rayos cercanos al eje; la teoría de Seidel lo predice con solo un rayo paraxial (ver `lentes/seidel.py`). Se prueba con diez lentes: caras esféricas ($k = 0$), elipsoidales ($k = 0.3$ y $-0.5$), parabólicas ($k = -1$) e hiperbólicas ($k = -1.5$ y $-3$), con el objeto cerca o en el infinito. Son los mismos casos de `tests/test_seidel.py`. Con $k = -3$ la lente queda sobrecorregida: $C_3 > 0$ y el borde enfoca después que el centro.

In [2]:
casos = [(10, 10, 2.0, -1.0, -30.0), (10, 10, 2.0, 0.0, -30.0), (8, 15, 3.0, 0.0, -40.0),
         (12, 6, 1.5, -0.5, -25.0), (10, 10, 2.0, -1.0, -60.0), (20, 7, 2.5, 0.3, -35.0),
         (10, 10, 2.0, 0.0, None), (15, 9, 2.0, -1.0, None), (10, 10, 2.0, -1.5, -30.0),
         (10, 10, 2.0, -3.0, -30.0)]
print(" R1   R2    d      k   objeto     C3 trazado    C3 Seidel   diferencia relativa")
diferencias = []
for R1, R2, d, k, y in casos:
    L = lente_biconvexa(R1, R2, d, 1.5, k, k)
    medido, teoria = coeficiente_esferica_trazado(L, y), coeficiente_esferica(L, y)
    diferencias.append(abs(medido / teoria - 1))
    objeto = "infinito" if y is None else f"{y:g}"
    print(f"{R1:3g} {R2:4g} {d:4g} {k:6g} {objeto:>8}   {medido:11.6f}  {teoria:11.6f}   {diferencias[-1]:.1e}")
print(f"\nMayor diferencia relativa: {max(diferencias):.1e}")

 R1   R2    d      k   objeto     C3 trazado    C3 Seidel   diferencia relativa
 10   10    2     -1      -30     -0.153448    -0.153448   5.1e-10
 10   10    2      0      -30     -0.268873    -0.268873   1.7e-09
  8   15    3      0      -40     -0.192958    -0.192958   4.1e-09
 12    6  1.5   -0.5      -25     -0.326309    -0.326309   1.5e-09
 10   10    2     -1      -60     -0.113522    -0.113522   1.5e-09
 20    7  2.5    0.3      -35     -0.415301    -0.415301   1.1e-08


 10   10    2      0 infinito     -0.146103    -0.146103   6.7e-09
 15    9    2     -1 infinito     -0.126118    -0.126118   2.9e-09
 10   10    2   -1.5      -30     -0.095735    -0.095735   1.6e-10
 10   10    2     -3      -30      0.077404     0.077404   4.5e-09

Mayor diferencia relativa: 1.1e-08


## 2. ¿Cuándo coincide con la paraxial? (actividad 4)

Altura de entrada a la que la diferencia llega al 1 % y al 0.1 %. El porcentaje se toma sobre la distancia imagen contada desde el centro de la lente ($y = 16.19$); el origen es una elección, por eso también se da $\Delta y$ en las unidades del modelo.

Un criterio físico, que no depende del origen, es el de Maréchal del [notebook 04](04_frente_de_onda.ipynb): con las longitudes en mm y luz verde, ¿hasta qué apertura la imagen en el plano paraxial sigue limitada por la difracción ($W_\text{rms} \le \lambda/14$)? Más allá, la diferencia con la paraxial ya se nota en la imagen.

Después, el espesor: cuánto se equivoca el **modelo de lente delgada** ($f$ de lente delgada y distancias medidas desde el centro) y cómo la misma ecuación $1/s + 1/s' = 1/f$ da la imagen exacta de la lente gruesa si se usa su focal efectiva y se miden $s$ y $s'$ desde los planos principales.

In [3]:
hs = np.geomspace(0.05, H_MAX, 400)
for nombre, L in (("parabólicas", parab), ("esféricas", esfer)):
    dy = aberracion_longitudinal(L, Y_OBJ, hs)
    rel = 100 * np.abs(dy) / y_paraxial(L, Y_OBJ)
    h1, h01 = np.interp(1.0, rel, hs), np.interp(0.1, rel, hs)
    print(f"caras {nombre}: C3 = {coeficiente_esferica(L, Y_OBJ):.4f};  1 % (Δy = {np.interp(h1, hs, np.abs(dy)):.3f})"
          f" en h = {h1:.2f} (h/R = {h1 / 10:.3f});  0.1 % en h = {h01:.2f};"
          f"  en el borde (h = {H_MAX:.2f}): Δy = {dy[-1]:.2f}, {rel[-1]:.1f} %")
    h_fis = apertura_limite(haz(L, Y_OBJ), plano="paraxial")
    print(f"   Maréchal en el plano paraxial (mm, 550 nm): hasta h = {h_fis:.2f}, donde Δy = "
          f"{np.interp(h_fis, hs, np.abs(dy)):.3f} ({np.interp(h_fis, hs, rel):.2f} %)")

ds = np.geomspace(0.01, 4.0, 400)
error = []
for d in ds:
    L = lente_biconvexa(10, 10, d, 1.5)
    gruesa, delgada = imagen_paraxial(L, (0, Y_OBJ)).y, imagen_lente_delgada(L, (0, Y_OBJ)).y
    error.append(100 * abs(delgada - gruesa) / gruesa)
error = np.array(error)
print(f"\nModelo delgado: error de 1 % con d = {np.interp(1.0, error, ds):.2f} (d/R = {np.interp(1.0, error, ds) / 10:.3f});"
      f" con d = 2 el error es {np.interp(2.0, ds, error):.1f} %")

y_gruesa, y_delgada = y_paraxial(parab, Y_OBJ), imagen_lente_delgada(parab, (0, Y_OBJ)).y
vertice = parab.arriba.y_vertice
print(f"Nuestra lente: paraxial y = {y_gruesa:.3f}, modelo delgado y = {y_delgada:.3f} (Δy = {y_gruesa - y_delgada:.3f});"
      f" contado desde el vértice de salida, el error es {100 * (y_gruesa - y_delgada) / (y_gruesa - vertice):.1f} %")

f = focal_efectiva(parab)
y_H, y_Hp = planos_principales(parab)
s = y_H - Y_OBJ
print(f"\nLente gruesa: focal efectiva f = {f:.3f}; planos principales en y = {y_H:.3f} y y = {y_Hp:.3f}")
print(f"1/s + 1/s' = 1/f con esa f, s desde H y s' desde H': y = {y_Hp + 1 / (1 / f - 1 / s):.4f}"
      f"  (matrices ABCD: {y_gruesa:.4f})")

caras parabólicas: C3 = -0.1534;  1 % (Δy = 0.162) en h = 1.03 (h/R = 0.103);  0.1 % en h = 0.32;  en el borde (h = 3.92): Δy = -2.47, 15.3 %


   Maréchal en el plano paraxial (mm, 550 nm): hasta h = 0.95, donde Δy = 0.140 (0.86 %)


caras esféricas: C3 = -0.2689;  1 % (Δy = 0.162) en h = 0.78 (h/R = 0.078);  0.1 % en h = 0.25;  en el borde (h = 3.92): Δy = -4.35, 26.9 %


   Maréchal en el plano paraxial (mm, 550 nm): hasta h = 0.83, donde Δy = 0.185 (1.14 %)

Modelo delgado: error de 1 % con d = 0.26 (d/R = 0.026); con d = 2 el error es 7.3 %
Nuestra lente: paraxial y = 16.187, modelo delgado y = 15.000 (Δy = 1.187); contado desde el vértice de salida, el error es 7.8 %

Lente gruesa: focal efectiva f = 10.345; planos principales en y = -0.310 y y = 0.310
1/s + 1/s' = 1/f con esa f, s desde H y s' desde H': y = 16.1872  (matrices ABCD: 16.1872)


## 3. Mejor foco (actividad 5)

Con aberración esférica no hay un punto donde se crucen todos los rayos. Se buscan el plano de radio RMS mínimo y el círculo de mínima confusión (el menor radio que encierra a todos los rayos, incluido el del borde). Para aberración de tercer orden pura la teoría da $2/3$ y $3/4$ del camino entre el foco paraxial y el del rayo más alejado.

In [4]:
from lentes.aberraciones import _pupila, abanico, radio_rms

for nombre, L in (("parabólicas", parab), ("esféricas", esfer)):
    for h_max in (0.3, H_MAX):
        f = mejores_focos(L, Y_OBJ, h_max)
        print(f"caras {nombre}, pupila h <= {h_max:.2f}: foco marginal y = {f.y_marginal:.3f};"
              f" RMS mínimo en y = {f.y_rms:.3f} (fracción {f.fraccion(f.y_rms):.3f}, radio {f.rms:.3f});"
              f" círculo en y = {f.y_circulo:.3f} (fracción {f.fraccion(f.y_circulo):.3f}, radio {f.radio_circulo:.3f})")
    f = mejores_focos(L, Y_OBJ, H_MAX)
    rms_paraxial = radio_rms(abanico(L, Y_OBJ, _pupila(H_MAX, 400)), f.y_paraxial)
    print(f"   en el plano paraxial el radio RMS es {rms_paraxial:.3f}: {rms_paraxial / f.rms:.1f} veces el del mejor foco\n")

caras parabólicas, pupila h <= 0.30: foco marginal y = 16.173; RMS mínimo en y = 16.178 (fracción 0.667, radio 0.000); círculo en y = 16.177 (fracción 0.750, radio 0.000)


caras parabólicas, pupila h <= 3.92: foco marginal y = 13.714; RMS mínimo en y = 14.523 (fracción 0.673, radio 0.113); círculo en y = 14.298 (fracción 0.764, radio 0.169)


   en el plano paraxial el radio RMS es 0.344: 3.0 veces el del mejor foco



caras esféricas, pupila h <= 0.30: foco marginal y = 16.163; RMS mínimo en y = 16.171 (fracción 0.667, radio 0.000); círculo en y = 16.169 (fracción 0.750, radio 0.000)


caras esféricas, pupila h <= 3.92: foco marginal y = 11.832; RMS mínimo en y = 13.199 (fracción 0.686, radio 0.214); círculo en y = 12.787 (fracción 0.781, radio 0.320)


   en el plano paraxial el radio RMS es 0.676: 3.2 veces el del mejor foco



## 4. Cómo se reduce: forma de la lente y caras cónicas

Forma: misma focal $f = 10$ y objeto en el infinito, variando el factor de forma $q$. Caras: la constante cónica que anula el tercer orden en la lente simétrica. La cónica de menor error de frente de onda y la tolerancia de fabricación se calculan en el [notebook 04](04_frente_de_onda.ipynb), porque necesitan el criterio de Maréchal.

In [5]:
for d in (2.0, 1e-4):
    r = minimize_scalar(lambda q: abs(coeficiente_esferica(lente_con_forma(q, 10.0, d), None)),
                        bounds=(0.2, 0.95), method="bounded", options={"xatol": 1e-7})
    valores = {q: abs(coeficiente_esferica(lente_con_forma(q, 10.0, d), None)) for q in (-1.0, 0.0, 1.0)}
    print(f"d = {d:g}: q óptimo = {r.x:.3f} (|C3| = {r.fun:.4f});  cara plana hacia la luz {valores[-1.0]:.4f},"
          f" simétrica {valores[0.0]:.4f}, cara curva hacia la luz {valores[1.0]:.4f}")
    print(f"   la forma óptima aberra {valores[-1.0] / r.fun:.1f} veces menos que la peor y"
          f" {valores[0.0] / r.fun:.1f} veces menos que la simétrica")
print(f"Lente delgada según los libros: q = 2(n² - 1)/(n + 2) = {2 * (1.5**2 - 1) / 3.5:.3f}")

k_cero = k_que_anula(esfer, Y_OBJ)
asf = lente_biconvexa(10, 10, 2.0, 1.5, k_cero, k_cero)
dy_esf, dy_asf = aberracion_longitudinal(esfer, Y_OBJ, [H_MAX])[0], aberracion_longitudinal(asf, Y_OBJ, [H_MAX])[0]
print(f"\nk que anula el tercer orden: {k_cero:.4f} (caras hiperbólicas)")
print(f"a h = {H_MAX:.2f}: esfera Δy = {dy_esf:.3f}, cónica Δy = {dy_asf:.4f} ({dy_esf / dy_asf:.0f} veces menos)")

pendiente = coeficiente_esferica(parab, Y_OBJ) - coeficiente_esferica(esfer, Y_OBJ)  # dC3/dk por unidad de -k
print(f"Sensibilidad: C3 cambia {abs(pendiente):.4f} por cada unidad de k")

d = 2: q óptimo = 0.648 (|C3| = 0.1012);  cara plana hacia la luz 0.4500, simétrica 0.1506, cara curva hacia la luz 0.1130
   la forma óptima aberra 4.4 veces menos que la peor y 1.5 veces menos que la simétrica
d = 0.0001: q óptimo = 0.714 (|C3| = 0.1071);  cara plana hacia la luz 0.4500, simétrica 0.1667, cara curva hacia la luz 0.1167
   la forma óptima aberra 4.2 veces menos que la peor y 1.6 veces menos que la simétrica
Lente delgada según los libros: q = 2(n² - 1)/(n + 2) = 0.714

k que anula el tercer orden: -2.3294 (caras hiperbólicas)
a h = 3.92: esfera Δy = -4.355, cónica Δy = -0.0283 (154 veces menos)
Sensibilidad: C3 cambia 0.1154 por cada unidad de k


## Resumen

- La teoría de Seidel predice el coeficiente de aberración del trazado con diferencias relativas del orden de $10^{-8}$ en las diez lentes, también con caras hiperbólicas.
- El trazado y la paraxial coinciden al 1 % solo con rayos a menos de una unidad del eje ($h/R \approx 0.1$). El modelo de lente delgada pide además $d/R$ de unas centésimas; con la focal efectiva y los planos principales, la misma ecuación $1/s + 1/s' = 1/f$ da la imagen paraxial exacta.
- El mejor foco no es el paraxial: está entre el foco paraxial y el del rayo más alejado, en las fracciones que predice la teoría.
- La forma de la lente y la forma de las caras cambian la aberración varias veces sin cambiar la focal.